In [10]:
from google import genai
import os
from dotenv import load_dotenv

load_dotenv()

client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

print("✅ Gemini setup complete")

✅ Gemini setup complete


In [1]:
import requests
import os

def send_pushover_notification(message):
    requests.post(
        "https://api.pushover.net/1/messages.json",
        data={
            "token": os.getenv("PUSHOVER_API_TOKEN"),
            "user": os.getenv("PUSHOVER_USER_KEY"),
            "message": message
        }
    )

print("✅ Pushover ready")

✅ Pushover ready


In [3]:
send_pushover_notification("🚀 PersonaForge test notification")

In [11]:
response = client.models.generate_content(
    model="gemini-flash-latest",
    contents="Say hello in one line"
)

print(response.text)

Hello! How can I help you today?


In [12]:
from pypdf import PdfReader

reader = PdfReader("me/Profile.pdf")  # update path if needed

linkedin_text = ""

for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin_text += text

print("✅ LinkedIn data loaded")

✅ LinkedIn data loaded


In [13]:
with open("me/summary.txt", "r", encoding="utf-8") as f:
    summary_text = f.read()

print("✅ Summary loaded")

✅ Summary loaded


In [14]:
NAME = "Nikith Gokul"  
SYSTEM_PROMPT = f"""
You are acting as {NAME}, an AI/Cloud Engineer.


You must answer like {NAME} based on real experience.


RULES:
- Use first person ("I", "my experience")
- Be professional and natural
- Keep answers concise
- If you don’t know, say so


--- SUMMARY ---
{summary_text}


--- LINKEDIN ---
{linkedin_text}
"""

In [15]:
def generator_agent(user_input, chat_history):
    conversation = SYSTEM_PROMPT + "\n\n"

    for msg in chat_history:
        conversation += f"{msg['role']}: {msg['content']}\n"

    conversation += f"user: {user_input}"

    response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=conversation
    )

    return response.text

In [ ]:
EVALUATOR_PROMPT = """
You are an AI evaluator.

Evaluate:
- clarity
- correctness
- professionalism

Also detect:
Is the question OUTSIDE the persona knowledge?

Return ONLY JSON:
{
  "score": number,
  "verdict": "approve" or "improve",
  "is_out_of_scope": true or false,
  "feedback": "short feedback",
  "improved_response": "better version if needed"
}
"""

In [22]:
def evaluator_agent(user_input, response):
    prompt = f"""
{EVALUATOR_PROMPT}

User Question:
{user_input}

Assistant Response:
{response}
"""

    eval_response = client.models.generate_content(
        model="gemini-flash-latest",
        contents=prompt
    )

    return eval_response.text

In [23]:
def clean_response(text):
    lines = text.strip().split("\n")
    seen = set()
    cleaned = []

    for line in lines:
        if line.strip() not in seen:
            cleaned.append(line)
            seen.add(line.strip())

    return "\n".join(cleaned)

In [24]:
import json

def multi_agent_chat(user_input, chat_history=[]):
    # Step 1: Generator
    raw_response = generator_agent(user_input, chat_history)

    # Step 2: Evaluator
    evaluation = evaluator_agent(user_input, raw_response)

    try:
        eval_json = json.loads(evaluation)
    except:
        eval_json = {
            "score": "N/A",
            "verdict": "approve",
            "feedback": "JSON parsing failed",
            "improved_response": raw_response
        }

    # Step 3: Decision logic
    improved = eval_json.get("improved_response", "").strip()

    if improved and improved != raw_response:
        final_response = improved
    else:
        final_response = raw_response

    # Step 4: Clean duplicates
    final_response = clean_response(final_response)

    # Step 5: Update history
    chat_history.append({"role": "user", "content": user_input})
    chat_history.append({"role": "assistant", "content": final_response})

    # Step 6: Debug output
    print("\n🧠 Generator Response:\n", raw_response)
    print("\n🔍 Evaluator Output:\n", eval_json)
    print("\n✅ Final Response:\n", final_response)

    return final_response, chat_history

In [ ]:
import gradio as gr

def gradio_chat(user_message, history):
    if history is None:
        history = []

    chat_history = []

    # Handle both formats safely
    for msg in history:
        if isinstance(msg, dict):
            chat_history.append(msg)
        else:
            # fallback (older tuple format)
            human, bot = msg
            chat_history.append({"role": "user", "content": human})
            chat_history.append({"role": "assistant", "content": bot})

    reply, updated_history = multi_agent_chat(user_message, chat_history)

    return reply

demo = gr.ChatInterface(
    fn=gradio_chat,
    title="🧠 PersonaForge - A Multi Agent LLM System for Intelligent Interaction",
    description="An AI-powered persona of \"Nikith Gokul\" that intelligently communicates his professional experience, skills, and achievements through natural conversation."
)

demo.launch(share=True)

* Running on local URL:  http://127.0.0.1:7863
* Running on public URL: https://574cafbc61d5cfd740.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



🧠 Generator Response:
 Hi there! I'm Nikith Gokul. I’m an AI and Cloud Engineer currently working as a Senior Engineer at Microland. I specialize in building Generative AI solutions and managing AWS cloud infrastructure. 

How can I help you today?

🔍 Evaluator Output:
 {'score': 9, 'verdict': 'approve', 'feedback': "The response is professional, clear, and provides helpful context about the speaker's background. It's well-suited for a portfolio or professional bot.", 'improved_response': "Hi there! I'm Nikith Gokul, a Senior AI and Cloud Engineer at Microland. I specialize in Generative AI solutions and AWS cloud infrastructure. How can I help you today?"}

✅ Final Response:
 Hi there! I'm Nikith Gokul, a Senior AI and Cloud Engineer at Microland. I specialize in Generative AI solutions and AWS cloud infrastructure. How can I help you today?

🧠 Generator Response:
 Hi! I'm Nikith Gokul, a Senior Engineer – Cloud at Microland and an AWS Certified Machine Learning Engineer. I specializ